
# ECF 2021 — Creación del Master Dataset amplio

Este notebook **sustituye** al generador anterior del master dataset.

## Criterio de construcción

1. Conserva las variables originales de los bloques analíticos.
2. Agrupa cada pregunta multirrespuesta en una sola columna padre terminada en `x`.
3. Elimina las columnas hijas después de comprobar que su información está en la columna padre.
4. Mantiene los **35 indicadores derivados** definidos en la versión anterior.
5. En el bloque de perfil conserva únicamente las variables demográficas y antecedentes útiles para segmentación o análisis.
6. Sobrescribe el master actual en `../Data/2026-07-20_ECF_2021_02_MasterDataset.csv`.

### Estructura esperada

```text
Proyecto/
├── Data/
│   └── 2026-07-20_ECF_2021_01_raw.dta
└── Scripts/
    └── 2026-07-20_ECF_2021_01_Create_Master_Dataset.ipynb
```


In [1]:

from pathlib import Path
from datetime import datetime
from collections import defaultdict
import re
import unicodedata

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 180)


## 1. Rutas y archivos

In [2]:

SCRIPTS_DIR = Path.cwd().resolve()
DATA_DIR = (SCRIPTS_DIR / "../Data").resolve()

RAW_PATH = DATA_DIR / "2026-07-20_ECF_2021_01_raw.dta"
OUTPUT_PATH = DATA_DIR / "2026-07-20_ECF_2021_02_MasterDataset.csv"
DICTIONARY_PATH = DATA_DIR / "2026-07-20_ECF_2021_02_MasterDataset_Diccionario.csv"
TRACE_PATH = DATA_DIR / "2026-07-20_ECF_2021_02_MasterDataset_Trazabilidad.csv"

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"No se encuentra el fichero raw: {RAW_PATH}\n"
        "Ejecuta este notebook desde la carpeta Scripts y comprueba ../Data/."
    )

DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Entrada : {RAW_PATH}")
print(f"Salida  : {OUTPUT_PATH}")


Entrada : /Users/rogerdefez/Documents/Cursos i Llibres/BootCamp IT Academy/04_Simulador/ProjecteData/Equip_32/Data/2026-07-20_ECF_2021_01_raw.dta
Salida  : /Users/rogerdefez/Documents/Cursos i Llibres/BootCamp IT Academy/04_Simulador/ProjecteData/Equip_32/Data/2026-07-20_ECF_2021_02_MasterDataset.csv


## 2. Carga del fichero original y metadatos

In [3]:

# Se cargan etiquetas para poder construir columnas padre legibles.
stata_reader = pd.io.stata.StataReader(RAW_PATH)
VARIABLE_LABELS = stata_reader.variable_labels()

raw_df = pd.read_stata(RAW_PATH, convert_categoricals=True)
RAW_COLUMNS = list(raw_df.columns)

print(f"Observaciones originales: {raw_df.shape[0]:,}")
print(f"Variables originales    : {raw_df.shape[1]:,}")
display(raw_df.head())


Observaciones originales: 7,764
Variables originales    : 429


,a01,a02,a0000,a0400,a04,a0800,a0100,a0320,a1030,a1040,a0910,a1100,a1200,a1300,a1400,a1410,a1420a,a1420b,a1420c,a1420d,a1420e,a1420f,a1420g,a1500,a1510,a1600,a1520,a1530,a2000,a1700,a1710,a1900,a2100,a2200,a2300,a2500,a2400,a2600,a2700,b0100,b0110a,b0110b,b0110c,b0110d,b0110e,b0110f,b0110g,b0120a,b0120b,b0120c,b0120d,b0120e,b0120f,b0130a,b0130b,b0130c,b0208,b0308,b0408,b0201,...,k0800,k0900,k1001,k1002,k1003,k1004,k1005,k1200,k1101,k0401,k0402,k0300,k1400,k1500,k1600,k1701,k1702,k1800,k1300,k2020,negativa_pmi,tmp_e0401,age,f0900a,f1000a,weight,a0940,a0930,a0300,a0310,ccaaf,a0930_b,a0950,f1100,f1200,f1200_b,f1300,a0900b,a0900c,a0900d,a0900e,a0900f,a0900l,a0900j,a0900h,a0900g,a0900i,a0900a,a0900k,a1800,a0200,f0900b,f1000b,f0900c,f1000c,f0900d,f1000d,f0900e,f1000e,ID
0,2022,6,Mujer,1982,40,No ha lugar (no aplica filtro),España,No,Casado,Gananciales,3,"g. Diplomaturas universitarias, grados universitarios de 240 créditos y similares","e. Otras ciencias sociales (Psicología, Sociología, Periodismo e Información...) y ciencias jurídicas (Derecho)",No ha lugar (no aplica filtro),d. Los suficientes para llenar dos estanterías (entre 101 y 200 libros),Sí,No mencionado,Mencionado,No mencionado,No mencionado,No mencionado,No mencionado,No mencionado,b. Trabaja por cuenta ajena,No tengo una segunda situación laboral,No ha lugar (no aplica filtro),A tiempo parcial,No ha lugar (no aplica filtro),Indefinido,No ha lugar (no aplica filtro),21,No,50,80,No ha lugar (no aplica filtro),c. Primera etapa de Educación Secundaria y similar,b. Trabaja por cuenta ajena,k. Dedicado a labores del hogar,"h. Operadores de instalaciones y maquinaria, y montadores",Sí,No mencionado,Mencionado,No mencionado,Mencionado,Mencionado,No mencionado,No mencionado,Mencionado,No mencionado,No mencionado,No mencionado,No mencionado,No mencionado,Mencionado,Mencionado,No mencionado,Sí,No,No,Sí,...,No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),69.70,40,Mencionado,Mencionado,6814.125977,Hombre,44.0,No ha lugar (no aplica filtro),-98.0,Andalucía,No ha lugar (no aplica filtro),15.0,Titular muestral,40.0,No ha lugar (no aplica filtro),Mujer,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,44.0,Andalucía,Mencionado,Mencionado,No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No mencionado,No mencionado,1.0
1,2022,3,Mujer,1967,54,No ha lugar (no aplica filtro),España,No,Soltero,No ha lugar (no aplica filtro),2,"h. Grados universitarios de más de 240 créditos, licenciaturas, másteres y similares","e. Otras ciencias sociales (Psicología, Sociología, Periodismo e Información...) y ciencias jurídicas (Derecho)",No ha lugar (no aplica filtro),a. Ninguno o muy pocos (entre 0 y 10 libros),No,No mencionado,Mencionado,Mencionado,Mencionado,Mencionado,No mencionado,No mencionado,b. Trabaja por cuenta ajena,No tengo una segunda situación laboral,No ha lugar (no aplica filtro),A tiempo completo,No ha lugar (no aplica filtro),No contesta,No ha lugar (no aplica filtro),27,No,20,10,No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),k. Dedicado a labores del hogar,"h. Operadores de instalaciones y maquinaria, y montadores",Sí,Mencionado,Mencionado,Mencionado,Mencionado,Mencionado,No mencionado,No mencionado,Mencionado,No mencionado,Mencionado,No mencionado,No mencionado,No mencionado,No 

## 3. Perfil y segmentación

In [4]:

# Variables del bloque A conservadas por su utilidad para segmentación,
# antecedentes financieros o análisis de competencias.
# La lista es explícita para evitar conservar variables administrativas
# que no aportan valor analítico.
SEGMENTATION_AND_BACKGROUND = [
    "ccaaf",      # Comunidad Autónoma
    "a0000",      # Sexo
    "a04",        # Edad calculada
    "a0100",      # País de nacimiento
    "a0320",      # Alguno de los padres nació fuera de España
    "a0910",      # Número de miembros del hogar
    "a1030",      # Estado civil
    "a1100",      # Nivel educativo
    "a1400",      # Libros en el hogar a los 10 años
    "a1410",      # Educación financiera recibida de los padres
    "a1500",      # Situación laboral principal
    "a1510",      # Situación laboral secundaria
    "a1520",      # Jornada completa/parcial
    "a1530",      # Personas a cargo
    "a1700",      # Experiencia laboral previa
    "a1710",      # Años trabajados
    "a1800",      # Ocupación
    "a1900",      # Trabajo relacionado con finanzas
]

missing_profile = [c for c in SEGMENTATION_AND_BACKGROUND if c not in raw_df.columns]
if missing_profile:
    print("⚠️ Variables de perfil no encontradas:", missing_profile)

SEGMENTATION_AND_BACKGROUND = [
    c for c in SEGMENTATION_AND_BACKGROUND if c in raw_df.columns
]


## 4. Creación de los mismos indicadores derivados

In [5]:

# ==========================================================
# FEATURE ENGINEERING — DATASET ANALÍTICO COMPACTO (65 VAR.)
# ==========================================================
# Esta sección parte de analytical_df (212 variables) y crea indicadores
# interpretables. Las variables originales no se modifican.

work = raw_df.copy()

# ---------- Utilidades ----------
NA_LABELS = {"No sabe", "No contesta", "-4", "-5", "No ha lugar (no aplica filtro)"}

def _norm_text(s):
    return s.astype("string").str.strip()

def es_mencionado(s):
    return _norm_text(s).eq("Mencionado")

def es_si(s):
    return _norm_text(s).eq("Sí")

def contar_mencionados(data, cols):
    """Cuenta opciones seleccionadas en preguntas de respuesta múltiple."""
    cols = [c for c in cols if c in data.columns]
    if not cols:
        return pd.Series(np.nan, index=data.index, dtype="float")
    return data[cols].apply(es_mencionado).sum(axis=1).astype("Int64")

def alguna_mencionada(data, cols):
    """1 cuando al menos una opción del grupo fue seleccionada."""
    return (contar_mencionados(data, cols) > 0).astype("Int64")

def alguna_si(data, cols):
    """1 cuando al menos una variable binaria del grupo toma el valor Sí."""
    cols = [c for c in cols if c in data.columns]
    if not cols:
        return pd.Series(pd.NA, index=data.index, dtype="Int64")
    return data[cols].apply(es_si).any(axis=1).astype("Int64")

def numeric_safe(s):
    return pd.to_numeric(s, errors="coerce")

def likert_score(s):
    """Extrae el valor inicial de etiquetas como '4-Con frecuencia'."""
    return pd.to_numeric(
        _norm_text(s).str.extract(r"^\s*(\d+)", expand=False),
        errors="coerce"
    )

def mean_likert(data, cols, reverse=None):
    """Media de ítems Likert; permite invertir ítems formulados negativamente."""
    reverse = set(reverse or [])
    values = pd.DataFrame(index=data.index)
    for c in cols:
        x = likert_score(data[c])
        if c in reverse:
            x = 6 - x
        values[c] = x
    return values.mean(axis=1, skipna=True)

def correct_score(series, accepted):
    """1 correcta; 0 incorrecta, no sabe o no contesta."""
    accepted = {str(v).strip() for v in accepted}
    return _norm_text(series).isin(accepted).astype("Int64")


# ==========================================================
# BLOQUE B — RELACIÓN BANCARIA, PRODUCTOS Y AHORRO
# ==========================================================
B0110 = [f"b0110{x}" for x in "abcdef"]
B0120 = [f"b0120{x}" for x in "abcdef"]
B030 = [
    "b0301", "b0302", "b0303", "b0304", "b0305", "b0306",
    "b0307", "b0308", "b0309", "b0310", "b0312"
]
B1000_ACTIVE = [f"b1000{x}" for x in "abcdefghi"]

work["n_canales_uso_banco"] = contar_mencionados(work, B0110)
work["usa_banca_digital"] = alguna_mencionada(
    work, ["b0110d", "b0110e"]
)
work["pref_banca_digital"] = alguna_mencionada(
    work, ["b0120d", "b0120e"]
)
work["usa_pago_digital"] = alguna_mencionada(
    work, ["b0130a", "b0130b"]
)

work["n_productos_financieros"] = (
    work[B030].apply(es_si).sum(axis=1).astype("Int64")
)
work["tiene_vehiculo_ahorro"] = alguna_si(
    work, ["b0302", "b0303", "b0304", "b0305", "b0308", "b0312"]
)
work["tiene_exposicion_credito"] = alguna_si(
    work, ["b0301", "b0306", "b0307"]
)
work["tiene_seguro"] = alguna_si(
    work, ["b0309", "b0310"]
)

work["n_vehiculos_ahorro"] = contar_mencionados(
    work, B1000_ACTIVE
)
work["ahorra_12m"] = (
    work["n_vehiculos_ahorro"].gt(0).astype("Int64")
)
# La opción explícita «no ha estado ahorrando» prevalece.
work.loc[
    es_mencionado(work["b1000j"]), "ahorra_12m"
] = 0

work["ahorro_formal"] = alguna_mencionada(
    work, ["b1000b", "b1000c", "b1000g", "b1000h"]
)
work["ahorro_informal"] = alguna_mencionada(
    work, ["b1000a", "b1000d", "b1000e", "b1000f", "b1000i"]
)

INDICATORS_B = [
    "n_canales_uso_banco", "usa_banca_digital", "pref_banca_digital",
    "usa_pago_digital", "n_productos_financieros", "tiene_vehiculo_ahorro",
    "tiene_exposicion_credito", "tiene_seguro", "ahorra_12m",
    "n_vehiculos_ahorro", "ahorro_formal", "ahorro_informal"
]
SCALARS_B = ["b0100", "b1201"]


# ==========================================================
# BLOQUE C — FUENTES DE INGRESO Y JUBILACIÓN
# ==========================================================
C0200 = [f"c0200{x}" for x in "abcdefghijk"]
C0400 = [f"c0400{x}" for x in "abcdefghijklmn"]
C0600 = [f"c0600{x}" for x in "abcdefghijk"]
C_ALL = C0200 + C0400 + C0600

work["n_fuentes_ingreso"] = contar_mencionados(work, C_ALL)
work["ingreso_por_activos"] = alguna_mencionada(
    work,
    [
        "c0200f", "c0200g", "c0200h",
        "c0400h", "c0400i", "c0400j", "c0400k", "c0400m",
        "c0600f", "c0600g", "c0600h"
    ]
)
work["dependencia_ingresos_familiares"] = alguna_mencionada(
    work,
    [
        "c0200d", "c0200e", "c0200i",
        "c0400c", "c0400d", "c0400e", "c0400f", "c0400l",
        "c0600d", "c0600e", "c0600i"
    ]
)

INDICATORS_C = [
    "n_fuentes_ingreso", "ingreso_por_activos",
    "dependencia_ingresos_familiares"
]
SCALARS_C = ["c0100", "c0300", "c0500"]


# ==========================================================
# BLOQUE D — ACTITUDES Y USO DE UN INGRESO EXTRA
# ==========================================================
# Constructos sin ítems duplicados:
# control/disciplina, orientación al futuro y preocupación financiera.
work["score_disciplina_financiera"] = mean_likert(
    work, ["d0101", "d0104", "d0106", "d0113"]
)
work["score_planificacion_financiera"] = mean_likert(
    work,
    ["d0102", "d0103", "d0107", "d0108", "d0116"],
    reverse=["d0102", "d0103", "d0108", "d0116"]
)
work["score_preocupacion_financiera"] = mean_likert(
    work, ["d0109", "d0110", "d0111", "d0114", "d0117"]
)

for c in ["d0610a", "d0610b", "d0610c", "d0610d", "d0610e"]:
    work[c] = numeric_safe(work[c])

work["puntos_destino_consumo"] = (
    work["d0610a"] + work["d0610b"]
)
work["puntos_destino_ahorro"] = work["d0610c"]
work["puntos_destino_deuda"] = work["d0610d"]

INDICATORS_D = [
    "score_disciplina_financiera", "score_preocupacion_financiera",
    "score_planificacion_financiera", "puntos_destino_consumo",
    "puntos_destino_ahorro", "puntos_destino_deuda"
]
SCALARS_D = ["d0300", "d0400", "d0500", "d0600"]


# ==========================================================
# BLOQUE E — COMPETENCIAS FINANCIERAS
# ==========================================================
work["score_alfabetizacion_financiera"] = (
    correct_score(
        work["e0600"],
        {"c. Menos de lo que podrían comprar hoy"}
    )
    + correct_score(work["e0900"], {"Más de 110 euros"})
    + correct_score(work["e1003"], {"Verdadero"})
).astype("Int64")

work["score_numeracy_financiera"] = (
    numeric_safe(work["e0500"]).eq(200).astype("Int64")
    + numeric_safe(work["e0700"]).eq(0).astype("Int64")
    + numeric_safe(work["e0800"]).eq(102).astype("Int64")
).astype("Int64")

work["score_comprension_riesgo"] = (
    correct_score(work["e1001"], {"Verdadero"})
    + correct_score(work["e1003"], {"Verdadero"})
    + correct_score(work["e1101"], {"Fondo 1 (Azul)"})
).astype("Int64")

work["score_competencia_economica"] = (
    correct_score(work["e1701"], {"Tipo de interés del 1%"})
    + correct_score(work["e1702"], {"Tipo de interés del 3%"})
    + correct_score(
        work["e1800"],
        {"a. La economía ha crecido en un 1%"}
    )
).astype("Int64")

work["score_conocimiento_productos"] = (
    correct_score(work["e1002"], {"Verdadero"})
    + correct_score(work["e1200"], {"Verdadero"})
).astype("Int64")

INDICATORS_E = [
    "score_alfabetizacion_financiera", "score_numeracy_financiera",
    "score_comprension_riesgo", "score_competencia_economica",
    "score_conocimiento_productos"
]
SCALARS_E = ["e0100", "e0300"]


# ==========================================================
# BLOQUE F — DECISIONES FINANCIERAS DEL HOGAR
# ==========================================================
SCALARS_F = ["f1100"]


# ==========================================================
# BLOQUE I — VIVIENDA
# ==========================================================
I0520 = [f"i0520{x}" for x in "abcdefg"]

work["barrera_acceso_vivienda"] = alguna_mencionada(
    work,
    ["i0200d", "i0200e", "i0200h", "i0520d", "i0520e", "i0520f"]
)
work["n_dificultades_compra_vivienda"] = contar_mencionados(
    work, I0520
)

for c in ["i0400a", "i0400b", "i0400c", "i0400d", "i0400e"]:
    work[c] = numeric_safe(work[c])

work["expectativa_subida_precio_vivienda"] = (
    work["i0400d"] + work["i0400e"]
)

INDICATORS_I = [
    "barrera_acceso_vivienda", "expectativa_subida_precio_vivienda",
    "n_dificultades_compra_vivienda"
]
SCALARS_I = ["i0100", "i0510", "i1000"]


# ==========================================================
# BLOQUE J — FRAGILIDAD FINANCIERA
# ==========================================================
work["financiacion_ahorros_activos"] = alguna_mencionada(
    work, ["j0300a", "j0300b"]
)
work["financiacion_credito_informal"] = alguna_mencionada(
    work, ["j0300c", "j0300d", "j0300l"]
)
work["financiacion_credito_formal"] = alguna_mencionada(
    work,
    [
        "j0300e", "j0300f", "j0300g", "j0300h",
        "j0300i", "j0300j", "j0300k"
    ]
)
work["financiacion_estres_pago"] = alguna_mencionada(
    work, ["j0300m", "j0300n"]
)
work["restriccion_acceso_credito"] = alguna_mencionada(
    work, ["j0900a", "j0900b", "j0900c"]
)

work["score_fragilidad_financiera"] = (
    es_si(work["j0200"]).astype("Int64")
    + es_si(work["j1000"]).astype("Int64")
    + es_si(work["j1201"]).astype("Int64")
    + work["restriccion_acceso_credito"]
).astype("Int64")

INDICATORS_J = [
    "financiacion_ahorros_activos", "financiacion_credito_informal",
    "financiacion_credito_formal", "financiacion_estres_pago",
    "restriccion_acceso_credito", "score_fragilidad_financiera"
]
SCALARS_J = [
    "j0100", "j0110", "j0200", "j0400",
    "j1000", "j1201", "j1203", "j1300"
]




## 5. Detección y agrupación de preguntas multirrespuesta

In [6]:

SPECIAL_RESPONSES = {
    "no sabe", "no contesta", "no ha lugar (no aplica filtro)",
    "-4", "-5", "nan", "<na>"
}
MENTIONED_VALUES = {"mencionado", "si", "sí", "1", "1.0", "true"}
NOT_MENTIONED_VALUES = {"no mencionado", "no", "0", "0.0", "false"}


def normalise_text(value):
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    text = "".join(
        ch for ch in unicodedata.normalize("NFD", text)
        if unicodedata.category(ch) != "Mn"
    )
    return text


def is_binary_multiresponse_column(series):
    values = {
        normalise_text(v)
        for v in series.dropna().unique()
    }
    allowed = {
        "mencionado", "no mencionado", "si", "no", "1", "1.0",
        "0", "0.0", "true", "false", "-4", "-5",
        "no sabe", "no contesta", "no ha lugar (no aplica filtro)"
    }
    return bool(values) and values.issubset(allowed)


def option_text(variable):
    """Extrae el texto de la opción a partir de la etiqueta Stata."""
    label = VARIABLE_LABELS.get(variable, variable)
    # Elimina el prefijo 'variable:' y después 'a.', 'b.', etc.
    label = re.sub(rf"^\s*{re.escape(variable)}\s*:\s*", "", label, flags=re.I)
    letter = variable[-1]
    match = re.search(rf"(?:^|[:?])\s*{letter}\s*[.)]\s*(.+)$", label, flags=re.I)
    if match:
        return match.group(1).strip()
    # Alternativa para etiquetas truncadas o con formato irregular.
    match = re.search(rf"\b{letter}\s*[.)]\s*(.+)$", label, flags=re.I)
    return match.group(1).strip() if match else label.strip()


def detect_multiresponse_groups(data):
    candidates = defaultdict(list)
    for column in data.columns:
        match = re.match(r"^([a-z]\d{4})([a-z])$", column.lower())
        if match:
            candidates[match.group(1)].append(column)

    groups = {}
    for base, columns in candidates.items():
        columns = sorted(columns)
        if len(columns) < 2:
            continue
        if all(is_binary_multiresponse_column(data[c]) for c in columns):
            groups[f"{base}x"] = columns
    return groups


def collapse_multiresponse(data, children):
    """Agrupa una multirrespuesta de forma vectorizada y legible."""
    result = pd.Series("", index=data.index, dtype="string")
    selected_any = pd.Series(False, index=data.index)

    mentioned_normalised = {normalise_text(v) for v in MENTIONED_VALUES}
    special_normalised = {normalise_text(v) for v in SPECIAL_RESPONSES}

    # Añadir las opciones seleccionadas respetando el orden de las hijas.
    for child in children:
        values = data[child].astype("string").map(normalise_text)
        selected = values.isin(mentioned_normalised)
        label = option_text(child)
        result.loc[selected & result.eq("")] = label
        result.loc[selected & result.ne("")] = (
            result.loc[selected & result.ne("")] + " | " + label
        )
        selected_any |= selected

    # Cuando no hay selección, conservar la primera respuesta especial disponible.
    unresolved = ~selected_any
    for child in children:
        if not unresolved.any():
            break
        raw_values = data[child].astype("string").str.strip()
        normalised = raw_values.map(normalise_text)
        special = unresolved & normalised.isin(special_normalised)
        result.loc[special] = raw_values.loc[special]
        unresolved &= ~special

    result.loc[result.eq("")] = "Ninguna opción mencionada"
    return result.astype("string")


MULTIRESPONSE_GROUPS = detect_multiresponse_groups(raw_df)
print(f"Preguntas multirrespuesta detectadas: {len(MULTIRESPONSE_GROUPS)}")

multiresponse_summary = pd.DataFrame([
    {
        "Variable_padre": parent,
        "N_hijas": len(children),
        "Variables_hijas": ", ".join(children),
        "Pregunta": re.sub(
            rf"^\s*{re.escape(children[0])}\s*:\s*", "",
            VARIABLE_LABELS.get(children[0], "")
        ).split(":")[0].strip(),
    }
    for parent, children in MULTIRESPONSE_GROUPS.items()
])

display(multiresponse_summary)


Preguntas multirrespuesta detectadas: 27


,Variable_padre,N_hijas,Variables_hijas,Pregunta
0,a1420x,7,"a1420a, a1420b, a1420c, a1420d, a1420e, a1420f, a1420g",ha recibido formación financiera de otra manera?
1,b0110x,7,"b0110a, b0110b, b0110c, b0110d, b0110e, b0110f, b0110g",en los ultimos 12 meses...
2,b0120x,6,"b0120a, b0120b, b0120c, b0120d, b0120e, b0120f",en que manera prefiera relacionarse con un banco
3,b0130x,3,"b0130a, b0130b, b0130c","en los últimos 12 meses, ha realizado algun pago…?"
4,b1100x,9,"b1100a, b1100b, b1100c, b1100d, b1100e, b1100f, b1100g, b1100h, b1100i",que productos son exclusivamente suyos?
5,b0720x,9,"b0720a, b0720b, b0720c, b0720d, b0720e, b0720f, b0720g, b0720h, b0720i",que fuentes de informacion utilizaron…
6,b0710x,9,"b0710a, b0710b, b0710c, b0710d, b0710e, b0710f, b0710g, b0710h, b0710i",que fuentes de informacion influyeron mas
7,b1000x,10,"b1000a, b1000b, b1000c, b1000d, b1000e, b1000f, b1000g, b1000h, b1000i, b1000j",ha estado los ultimos 12 meses
8,b1203x,7,"b1203a, b1203b, b1203c, b1203d, b1203e, b1203f, b1203g",con que productos tuve desavenencias?
9,b1204x,7,"b1204a, b1204b, b1204c, b1204d, b1204e, b1204f, b1204g",que tipo de desavenencia sobre prestamos bancarios?


## 6. Construcción del nuevo master amplio

In [7]:

# 1) Mantener todos los bloques B–J y las variables de perfil seleccionadas.
NON_PROFILE_COLUMNS = [
    c for c in RAW_COLUMNS
    if not c.lower().startswith("a") and c != "ccaaf"
]

BASE_COLUMNS_TO_KEEP = list(dict.fromkeys(
    SEGMENTATION_AND_BACKGROUND + NON_PROFILE_COLUMNS
))

master_df = work[BASE_COLUMNS_TO_KEEP].copy()

# 2) Crear columnas padre antes de eliminar las hijas.
for parent, children in MULTIRESPONSE_GROUPS.items():
    available_children = [c for c in children if c in work.columns]
    if available_children:
        master_df[parent] = collapse_multiresponse(work, available_children)

# 3) Eliminar todas las hijas multirrespuesta del master.
MULTIRESPONSE_CHILDREN = sorted({
    child
    for children in MULTIRESPONSE_GROUPS.values()
    for child in children
})
master_df.drop(
    columns=[c for c in MULTIRESPONSE_CHILDREN if c in master_df.columns],
    inplace=True,
)

# 4) Añadir todos los indicadores derivados, incluso cuando sus fuentes
# hayan sido sustituidas por una columna padre.
ALL_INDICATORS = (
    INDICATORS_B + INDICATORS_C + INDICATORS_D + INDICATORS_E
    + INDICATORS_I + INDICATORS_J
)

for indicator in ALL_INDICATORS:
    master_df[indicator] = work[indicator]

print(f"Observaciones del nuevo master: {master_df.shape[0]:,}")
print(f"Variables del nuevo master    : {master_df.shape[1]:,}")


Observaciones del nuevo master: 7,764
Variables del nuevo master    : 255


## 7. Validaciones de integridad

In [8]:

# Validaciones estructurales.
assert master_df.shape[0] == raw_df.shape[0], "Ha cambiado el número de registros."
assert not master_df.columns.duplicated().any(), "Hay columnas duplicadas."
assert not master_df.columns.isna().any(), "Hay nombres de columna nulos."

remaining_children = [c for c in MULTIRESPONSE_CHILDREN if c in master_df.columns]
assert not remaining_children, (
    f"Quedan hijas multirrespuesta en el master: {remaining_children}"
)

missing_parents = [p for p in MULTIRESPONSE_GROUPS if p not in master_df.columns]
assert not missing_parents, f"Faltan variables padre: {missing_parents}"

missing_indicators = [c for c in ALL_INDICATORS if c not in master_df.columns]
assert not missing_indicators, f"Faltan indicadores: {missing_indicators}"

# Cada fila padre debe contener texto no vacío.
empty_parent_counts = {
    parent: int(master_df[parent].isna().sum() + master_df[parent].eq("").sum())
    for parent in MULTIRESPONSE_GROUPS
}
assert not any(empty_parent_counts.values()), (
    f"Hay padres vacíos: {empty_parent_counts}"
)

# Las variables originales no demográficas y no multirrespuesta deben seguir presentes.
expected_direct_originals = [
    c for c in NON_PROFILE_COLUMNS
    if c not in MULTIRESPONSE_CHILDREN
]
missing_direct_originals = [
    c for c in expected_direct_originals if c not in master_df.columns
]
assert not missing_direct_originals, (
    "Se han perdido variables originales no multirrespuesta: "
    f"{missing_direct_originals}"
)

print("✅ Todas las validaciones estructurales han sido superadas.")
print(f"Variables hijas eliminadas : {len(MULTIRESPONSE_CHILDREN)}")
print(f"Variables padre creadas    : {len(MULTIRESPONSE_GROUPS)}")
print(f"Indicadores conservados    : {len(ALL_INDICATORS)}")
print(f"Columnas totalmente vacías : {int(master_df.isna().all().sum())}")


✅ Todas las validaciones estructurales han sido superadas.
Variables hijas eliminadas : 212
Variables padre creadas    : 27
Indicadores conservados    : 35
Columnas totalmente vacías : 0


## 8. Trazabilidad de la transformación

In [9]:

trace_rows = []

for column in RAW_COLUMNS:
    if column in MULTIRESPONSE_CHILDREN:
        parent = next(
            p for p, children in MULTIRESPONSE_GROUPS.items()
            if column in children
        )
        status = "Agrupada en variable padre"
        destination = parent
    elif column in master_df.columns:
        status = "Conservada directamente"
        destination = column
    elif column.lower().startswith("a") or column == "ccaaf":
        status = "Eliminada: perfil no seleccionado para segmentación"
        destination = ""
    else:
        status = "Revisar"
        destination = ""

    trace_rows.append({
        "Variable_original": column,
        "Etiqueta_Stata": VARIABLE_LABELS.get(column, ""),
        "Estado": status,
        "Variable_destino": destination,
    })

for indicator in ALL_INDICATORS:
    trace_rows.append({
        "Variable_original": "",
        "Etiqueta_Stata": "Indicador derivado",
        "Estado": "Indicador añadido",
        "Variable_destino": indicator,
    })

traceability_df = pd.DataFrame(trace_rows)
summary_df = (
    traceability_df["Estado"]
    .value_counts()
    .rename_axis("Estado")
    .reset_index(name="N_variables")
)

display(summary_df)
display(traceability_df.head(20))


,Estado,N_variables
0,Agrupada en variable padre,212
1,Conservada directamente,193
2,Indicador añadido,35
3,Eliminada: perfil no seleccionado para segmentación,24


,Variable_original,Etiqueta_Stata,Estado,Variable_destino
0,a01,a01: anio de la entrevista,Eliminada: perfil no seleccionado para segmentación,
1,a02,a02: mes de la entrevista,Eliminada: perfil no seleccionado para segmentación,
2,a0000,"a0000: es requisito que pregunte su genero, es vd. hombre o mujer?",Conservada directamente,a0000
3,a0400,a0400: en que anio nacio?,Eliminada: perfil no seleccionado para segmentación,
4,a04,a04: edad calculada,Conservada directamente,a04
5,a0800,a0800: me podria decir su edad aproximada?,Eliminada: perfil no seleccionado para segmentación,
6,a0100,a0100: en que pais nacio?,Conservada directamente,a0100
7,a0320,a0320: nacio alguno de sus padres fuera de espania?,Conservada directamente,a0320
8,a1030,a1030: cual es su estado civil actual?,Conservada directamente,a1030
9,a1040,a1040: cual es su regimen economico matrimonial?,Eliminada: perfil no seleccionado para segmentación,


## 9. Diccionario de indicadores

In [10]:

# ==========================================================
# DICCIONARIO DE INDICADORES CREADOS
# ==========================================================

INDICATOR_DICTIONARY = pd.DataFrame([
    ("n_canales_uso_banco", "Número de canales utilizados para relacionarse con el banco."),
    ("usa_banca_digital", "Uso de ordenador/tablet o app móvil bancaria."),
    ("pref_banca_digital", "Preferencia por ordenador/tablet o app móvil bancaria."),
    ("usa_pago_digital", "Uso de aplicaciones o banca online para realizar pagos."),
    ("n_productos_financieros", "Número de familias/productos financieros actualmente contratados."),
    ("tiene_vehiculo_ahorro", "Tenencia de algún producto de ahorro o inversión."),
    ("tiene_exposicion_credito", "Tenencia de hipoteca, préstamo personal o tarjeta de crédito."),
    ("tiene_seguro", "Tenencia de seguro de vida o seguro médico."),
    ("ahorra_12m", "Ha utilizado al menos un mecanismo de ahorro en los últimos 12 meses."),
    ("n_vehiculos_ahorro", "Número de mecanismos de ahorro utilizados."),
    ("ahorro_formal", "Uso de cuentas, depósitos, fondos o planes de pensiones para ahorrar."),
    ("ahorro_informal", "Uso de efectivo, familia, inmuebles, remesas u otros mecanismos."),
    ("n_fuentes_ingreso", "Número de fuentes actuales o previstas de ingresos."),
    ("ingreso_por_activos", "Obtiene o prevé obtener ingresos mediante activos o ahorros."),
    ("dependencia_ingresos_familiares", "Dependencia de pareja, familia, ayudas o instituciones."),
    ("score_disciplina_financiera", "Media 1–5 de control, pago puntual, vigilancia e información."),
    ("score_preocupacion_financiera", "Media 1–5 de preocupación, endeudamiento e inquietud."),
    ("score_planificacion_financiera", "Media 1–5 orientada al futuro; ítems negativos invertidos."),
    ("puntos_destino_consumo", "Puntos del ingreso extra destinados a consumo."),
    ("puntos_destino_ahorro", "Puntos del ingreso extra destinados a ahorro."),
    ("puntos_destino_deuda", "Puntos del ingreso extra destinados a pagar deuda."),
    ("score_alfabetizacion_financiera", "Aciertos 0–3: inflación, interés compuesto y diversificación."),
    ("score_numeracy_financiera", "Aciertos 0–3 en cálculos financieros básicos."),
    ("score_comprension_riesgo", "Aciertos 0–3 sobre riesgo, diversificación y rendimiento."),
    ("score_competencia_economica", "Aciertos 0–3 sobre tipos de interés y crecimiento económico."),
    ("score_conocimiento_productos", "Aciertos 0–2 sobre inflación y características hipotecarias."),
    ("barrera_acceso_vivienda", "Presenta barreras de entrada, cuota o acceso hipotecario."),
    ("expectativa_subida_precio_vivienda", "Puntos asignados a escenarios de subida del precio."),
    ("n_dificultades_compra_vivienda", "Número de dificultades experimentadas al comprar vivienda."),
    ("financiacion_ahorros_activos", "Cubrió el déficit utilizando ahorros o vendiendo activos."),
    ("financiacion_credito_informal", "Cubrió el déficit mediante familia, adelantos o proveedores."),
    ("financiacion_credito_formal", "Cubrió el déficit mediante crédito o financiación formal."),
    ("financiacion_estres_pago", "Utilizó descubierto no autorizado o retrasó pagos."),
    ("restriccion_acceso_credito", "Rechazo, concesión parcial o autoexclusión crediticia."),
    ("score_fragilidad_financiera", "Índice 0–4: déficit, impagos, pérdida de empleo y restricción crediticia."),
], columns=["Variable", "Definición"])

display(INDICATOR_DICTIONARY)


,Variable,Definición
0,n_canales_uso_banco,Número de canales utilizados para relacionarse con el banco.
1,usa_banca_digital,Uso de ordenador/tablet o app móvil bancaria.
2,pref_banca_digital,Preferencia por ordenador/tablet o app móvil bancaria.
3,usa_pago_digital,Uso de aplicaciones o banca online para realizar pagos.
4,n_productos_financieros,Número de familias/productos financieros actualmente contratados.
5,tiene_vehiculo_ahorro,Tenencia de algún producto de ahorro o inversión.
6,tiene_exposicion_credito,"Tenencia de hipoteca, préstamo personal o tarjeta de crédito."
7,tiene_seguro,Tenencia de seguro de vida o seguro médico.
8,ahorra_12m,Ha utilizado al menos un mecanismo de ahorro en los últimos 12 meses.
9,n_vehiculos_ahorro,Número de mecanismos de ahorro utilizados.


## 10. Exportación — sustituye el master actual

In [11]:

# La exportación usa exactamente el mismo nombre para sustituir el master actual.
master_df.to_csv(OUTPUT_PATH, index=False)
INDICATOR_DICTIONARY.to_csv(DICTIONARY_PATH, index=False)
traceability_df.to_csv(TRACE_PATH, index=False)

print("✅ Archivos exportados correctamente")
print(f"Master        : {OUTPUT_PATH}")
print(f"Diccionario   : {DICTIONARY_PATH}")
print(f"Trazabilidad  : {TRACE_PATH}")
print(f"Dimensiones   : {master_df.shape}")


✅ Archivos exportados correctamente
Master        : /Users/rogerdefez/Documents/Cursos i Llibres/BootCamp IT Academy/04_Simulador/ProjecteData/Equip_32/Data/2026-07-20_ECF_2021_02_MasterDataset.csv
Diccionario   : /Users/rogerdefez/Documents/Cursos i Llibres/BootCamp IT Academy/04_Simulador/ProjecteData/Equip_32/Data/2026-07-20_ECF_2021_02_MasterDataset_Diccionario.csv
Trazabilidad  : /Users/rogerdefez/Documents/Cursos i Llibres/BootCamp IT Academy/04_Simulador/ProjecteData/Equip_32/Data/2026-07-20_ECF_2021_02_MasterDataset_Trazabilidad.csv
Dimensiones   : (7764, 255)


## 11. Comprobación del archivo exportado

In [12]:

exported_header = pd.read_csv(OUTPUT_PATH, nrows=0)
assert list(exported_header.columns) == list(master_df.columns), (
    "Las columnas exportadas no coinciden con el master en memoria."
)
assert OUTPUT_PATH.stat().st_size > 0, "El CSV exportado está vacío."

print("✅ Exportación validada")
print(f"Fecha de generación: {datetime.now():%Y-%m-%d %H:%M:%S}")
display(master_df.head())


✅ Exportación validada
Fecha de generación: 2026-07-22 14:02:46


,ccaaf,a0000,a04,a0100,a0320,a0910,a1030,a1100,a1400,a1410,a1500,a1510,a1520,a1530,a1700,a1710,a1800,a1900,b0100,b0208,b0308,b0408,b0201,b0301,b0401,b0202,b0302,b0402,b0203,b0303,b0403,b0204,b0304,b0404,b0205,b0305,b0405,b0206,b0306,b0406,b0207,b0307,b0407,b0209,b0309,b0409,b0210,b0310,b0410,b0212,b0312,b0412,b0502,b0503,b0600,b1103,b1201,c0100,c0300,c0500,...,b0120x,b0130x,b1100x,b0720x,b0710x,b1000x,b1203x,b1204x,b1205x,b1206x,b1207x,b1208x,b1209x,b1230x,c0200x,c0600x,c0400x,i0200x,i0500x,i0520x,j0300x,j0800x,j0900x,j0910x,a0900x,n_canales_uso_banco,usa_banca_digital,pref_banca_digital,usa_pago_digital,n_productos_financieros,tiene_vehiculo_ahorro,tiene_exposicion_credito,tiene_seguro,ahorra_12m,n_vehiculos_ahorro,ahorro_formal,ahorro_informal,n_fuentes_ingreso,ingreso_por_activos,dependencia_ingresos_familiares,score_disciplina_financiera,score_preocupacion_financiera,score_planificacion_financiera,puntos_destino_consumo,puntos_destino_ahorro,puntos_destino_deuda,score_alfabetizacion_financiera,score_numeracy_financiera,score_comprension_riesgo,score_competencia_economica,score_conocimiento_productos,barrera_acceso_vivienda,expectativa_subida_precio_vivienda,n_dificultades_compra_vivienda,financiacion_ahorros_activos,financiacion_credito_informal,financiacion_credito_formal,financiacion_estres_pago,restriccion_acceso_credito,score_fragilidad_financiera
0,Andalucía,Mujer,40,España,No,3,Casado,"g. Diplomaturas universitarias, grados universitarios de 240 créditos y similares",d. Los suficientes para llenar dos estanterías (entre 101 y 200 libros),Sí,b. Trabaja por cuenta ajena,No tengo una segunda situación laboral,A tiempo parcial,No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),21,44.0,No,Sí,Sí,No,No,Sí,Sí,No,Sí,No,No,Sí,No,No,Sí,No,No,Sí,No,No,Sí,Sí,Sí,Sí,Sí,No,Sí,No,No,Sí,No,No,Sí,No,No,Préstamo personal,Conjunta,b. Consideré/ consideramos varias opciones de una única empresa o institución,No,No,1 (MUY MAL),60,No ha lugar (no aplica filtro),...,visitando personalm | visitando personalm,utilizando una ap | utilizando una ap | utilizando apps ba,ninguno | ninguno,informacion proporcionada por | informacion proporcionada por,informacion proporcionada | informacion proporcionada,ahorrando dinero en metalico | ahorrando dinero en metalico,No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),juzgado | juzgado | organiz,pension publica | pension publica | ingresos de un conyuge o pareja,No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),es mas barato ser propietario que alquilar | es mas barato ser propietario que alquilar | compre mi vivienda como inversion,No ha lugar (no aplica filtro),utilizar ahorros | utilizar ahorros,hijos menores de 18 anios | hijos menores de 18 anios,"no, ninguna de las anteriores | no, ninguna de las anteriores",No ha lugar (no aplica filtro),con su pareja | con su pareja,3,1,0,1,3,0,1,0,1,1,0,1,2,0,1,3.75,4.4,3.2,5.0,3.0,1.0,0,2,2,3,2,0,2.0,0,1,0,0,0,0,1
1,Canarias,Mujer,54,España,No,2,Soltero,"h. Grados universitarios de más de 240 créditos, licenciaturas, másteres y similares",a. Ninguno o muy pocos (entre 0 y 10 libros),No,b. Trabaja por cuenta ajena,No tengo una segunda situación laboral,A tiempo completo,No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),27,22.0,No,Sí,Sí,No,No,Sí,No,No,Sí,No,No,Sí,No,No,Sí,No,No,Sí,No,No,Sí,Sí,No,Sí,Sí,No,Sí,No,No,Sí,Sí,No,Sí,No,No,No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),No,No,3 (REGULAR),67,No ha lugar (no aplica filtro),...,visitando personalm | visitando personalm | llamando por telefo,utilizando apps ba | utilizando apps ba,cuenta corriente | cuenta corriente | prestamos,No ha lugar (no aplica filtro),No ha lugar (no aplica filtro),no ha estado ahorrando | no ha estado ahorrando,N